# Этап 6 V2 — проверка вычислительной реализуемости TabM

## Исследовательский вопрос

Может ли облегчённая конфигурация TabM завершить обучение на KOMUS за разумное время и дать первичную оценку качества?

## Зачем сейчас нужна проверка реализуемости

Этап 6 V1 с `k=32`, тремя внешними фолдами и лимитом 1000 эпох не завершился за восемь часов и не дал метрик. Поэтому до повторного полного запуска требуется отдельный короткий диагностический прогон: он отвечает на вопрос о скорости и завершении обучения, а не о превосходстве над GBDT.

## Что проверяем

Запускается только первый фолд заранее зафиксированного `StratifiedKFold`. У TabM уменьшено только `k` до 8 и ограничено число эпох до 100; подбор и поиск параметров отсутствуют. Прогон ограничен двумя часами, чтобы диагностическая проверка не стала неконтролируемой вычислительной задачей.

## Что остаётся неизменным

Используются `Data_final.xlsb`, целевой признак `DefMark`, идентификатор `INN`, те же 47 разрешённых признаков в том же порядке, исключение `Q_B1_norm` и `Q_B2_norm`, проверка SHA-256, предобработка без заполнения пропусков, фиксированное разбиение 80/20 с зерном случайности 42 и зерном 43 для первого фолда. Закрытая 20%-выборка создаётся только как часть воспроизводимого разбиения и немедленно освобождается: она не участвует ни в обучении, ни в прогнозе, ни в метриках.

Этот блокнот не заменяет Этап 6 V1, не выполняет финальное сравнение моделей и не сохраняет новые отчётные артефакты.

## Подготовка контракта эксперимента

1. **Что проверяем?** Доступность данных, точную идентичность 47 признаков и параметры облегчённого запуска.
2. **Зачем это делаем сейчас?** Диагностика имеет смысл только при сохранении исходного исследовательского контракта.
3. **Как этот код отвечает на исследовательский вопрос?** Он загружает список признаков из принятого базового результата Этапа 1 и задаёт единственную облегчающую конфигурацию.
4. **Что остаётся неизменным?** Данные, целевой признак, исключённые признаки, порядок признаков и зерна случайности.

In [1]:
from __future__ import annotations

import hashlib
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tabm
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset


def project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Не найден корень проекта с pyproject.toml.')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def hash_indices(values: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def set_fold_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)


ROOT = project_root()
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
BASELINE_PATH = ROOT / 'reports' / 'generated' / 'stage1_baseline_results_V2.json'
EXPECTED_DATASET_SHA256 = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_INDEX_SHA256 = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
TARGET = 'DefMark'
IDENTIFIER = 'INN'
FORBIDDEN_FEATURES = ['Q_B1_norm', 'Q_B2_norm']
OUTER_SEED = 42
OUTER_FOLD_NUMBER = 1
FOLD_SEED = 43
MAX_EPOCHS = 100
PATIENCE = 16
RUNTIME_BUDGET_SECONDS = 2 * 60 * 60
PROGRESS_EVERY_BATCHES = 10
TABM_CONFIG = {
    'arch_type': 'tabm', 'k': 8, 'n_blocks': 3, 'd_block': 512,
    'activation': 'ReLU', 'dropout': 0.10, 'start_scaling_init': 'random-signs',
    'num_embeddings': None, 'd_out': 2, 'input_dtype': 'float32',
    'optimizer': 'AdamW', 'lr': 0.002, 'weight_decay': 0.0003,
    'betas': (0.9, 0.999), 'eps': 1e-8, 'gradient_clip_global_norm': 1.0,
    'batch_size': 256, 'share_training_batches': True, 'max_epochs': MAX_EPOCHS,
    'amp': False, 'torch_compile': False, 'scheduler': None, 'warmup': None,
    'class_weights': None, 'sampling': None,
}

with BASELINE_PATH.open(encoding='utf-8') as handle:
    baseline = json.load(handle)
FEATURES = baseline['допустимые_признаки']
if len(FEATURES) != 47 or any(name in FEATURES for name in FORBIDDEN_FEATURES):
    raise ValueError('Нарушен контракт Stage 1: требуется ровно 47 разрешённых признаков.')

print({'этап': 'конфигурация', 'признаков': len(FEATURES), 'k': TABM_CONFIG['k'], 'максимум_эпох': MAX_EPOCHS, 'внешний_фолд': OUTER_FOLD_NUMBER, 'seed_фолда': FOLD_SEED})

{'этап': 'конфигурация', 'признаков': 47, 'k': 8, 'максимум_эпох': 100, 'внешний_фолд': 1, 'seed_фолда': 43}


## Обучение с наблюдаемым прогрессом

1. **Что проверяем?** Способно ли обучение с `k=8` продвигаться по эпохам и завершить отбор эпохи и дообучение.
2. **Зачем это делаем сейчас?** Именно отсутствие наблюдаемого завершения стало ограничением V1.
3. **Как этот код отвечает на исследовательский вопрос?** До и после долгих операций, каждые 10 батчей и после каждой эпохи выводятся этап, фолд, число эпох, минуты, лучший ROC-AUC и оценка оставшегося времени.
4. **Что остаётся неизменным?** Архитектурные параметры, кроме `k`, оптимизатор, размер батча, ранняя остановка, способ выбора эпохи и последующее дообучение соответствуют V1.

In [2]:
class RuntimeBudgetExceeded(RuntimeError):
    pass


def elapsed_seconds(started_at: float) -> float:
    return time.perf_counter() - started_at


def check_budget(started_at: float) -> None:
    if elapsed_seconds(started_at) > RUNTIME_BUDGET_SECONDS:
        raise RuntimeBudgetExceeded('Превышен двухчасовой лимит диагностического запуска.')


def show_progress(stage: str, epoch: int | None, total_epochs: int | None, started_at: float, best_auc: float | None, batch: int | None = None, batches: int | None = None, estimated_remaining_seconds: float | None = None) -> None:
    message = {
        'этап': stage, 'фолд': OUTER_FOLD_NUMBER,
        'текущая_эпоха': epoch, 'максимум_эпох': total_epochs,
        'эпоха': None if epoch is None or total_epochs is None else f'{epoch}/{total_epochs}',
        'прошло_минут': round(elapsed_seconds(started_at) / 60.0, 2),
        'лучший_ROC_AUC': None if best_auc is None or not np.isfinite(best_auc) else round(best_auc, 6),
        'примерно_осталось_минут': None if estimated_remaining_seconds is None else round(max(0.0, estimated_remaining_seconds) / 60.0, 2),
    }
    if batch is not None and batches is not None:
        message['батч'] = f'{batch}/{batches}'
    print(message, flush=True)


def show_stage(stage: str, started_at: float, best_auc: float | None = None) -> None:
    show_progress(stage, None, MAX_EPOCHS, started_at, best_auc)


def make_model(n_features: int) -> tabm.TabM:
    return tabm.TabM.make(
        n_num_features=n_features, cat_cardinalities=None,
        arch_type=TABM_CONFIG['arch_type'], k=TABM_CONFIG['k'],
        n_blocks=TABM_CONFIG['n_blocks'], d_block=TABM_CONFIG['d_block'],
        activation=TABM_CONFIG['activation'], dropout=TABM_CONFIG['dropout'],
        start_scaling_init=TABM_CONFIG['start_scaling_init'],
        num_embeddings=None, d_out=TABM_CONFIG['d_out'],
    )


def make_loader(x: np.ndarray, y: np.ndarray, seed: int, shuffle: bool) -> DataLoader:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y.astype(np.int64)))
    return DataLoader(dataset, batch_size=TABM_CONFIG['batch_size'], shuffle=shuffle, generator=generator, num_workers=0, drop_last=False)


def fit_one_epoch(model: tabm.TabM, loader: DataLoader, optimizer: torch.optim.Optimizer, started_at: float, stage: str, epoch: int, total_epochs: int, best_auc: float | None) -> None:
    model.train()
    batches = len(loader)
    epoch_started_at = time.perf_counter()
    for batch_number, (x_batch, y_batch) in enumerate(loader, start=1):
        check_budget(started_at)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x_batch.float())
        targets = y_batch[:, None].expand(-1, model.k).reshape(-1)
        loss = F.cross_entropy(logits.reshape(-1, 2), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), TABM_CONFIG['gradient_clip_global_norm'])
        optimizer.step()
        if batch_number == 1 or batch_number % PROGRESS_EVERY_BATCHES == 0 or batch_number == batches:
            batch_seconds = (time.perf_counter() - epoch_started_at) / batch_number
            remaining_batches = (batches - batch_number) + max(0, total_epochs - epoch) * batches
            show_progress(stage, epoch, total_epochs, started_at, best_auc, batch_number, batches, batch_seconds * remaining_batches)


@torch.inference_mode()
def positive_probabilities(model: tabm.TabM, x: np.ndarray, started_at: float, stage: str, epoch: int, total_epochs: int, best_auc: float | None) -> np.ndarray:
    model.eval()
    tensor = torch.from_numpy(x).float()
    chunks: list[np.ndarray] = []
    starts = list(range(0, len(tensor), TABM_CONFIG['batch_size']))
    prediction_started_at = time.perf_counter()
    for chunk_number, start in enumerate(starts, start=1):
        check_budget(started_at)
        logits = model(tensor[start:start + TABM_CONFIG['batch_size']])
        chunks.append(torch.softmax(logits, dim=-1).mean(dim=1)[:, 1].cpu().numpy())
        if chunk_number == 1 or chunk_number % PROGRESS_EVERY_BATCHES == 0 or chunk_number == len(starts):
            chunk_seconds = (time.perf_counter() - prediction_started_at) / chunk_number
            show_progress(stage, epoch, total_epochs, started_at, best_auc, chunk_number, len(starts), chunk_seconds * (len(starts) - chunk_number))
    return np.concatenate(chunks)


def make_optimizer(model: tabm.TabM) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=TABM_CONFIG['lr'], weight_decay=TABM_CONFIG['weight_decay'], betas=TABM_CONFIG['betas'], eps=TABM_CONFIG['eps'])


def select_best_epoch(x_outer: np.ndarray, y_outer: np.ndarray, started_at: float) -> tuple[int, float]:
    show_stage('выбор эпохи: подготовка', started_at)
    fit_idx, inner_idx = train_test_split(np.arange(len(y_outer)), test_size=0.10, stratify=y_outer, random_state=FOLD_SEED)
    set_fold_seed(FOLD_SEED)
    model = make_model(x_outer.shape[1])
    optimizer = make_optimizer(model)
    loader = make_loader(x_outer[fit_idx], y_outer[fit_idx], FOLD_SEED, shuffle=True)
    best_epoch, best_auc, stale_epochs = 0, float('-inf'), 0
    epoch_durations: list[float] = []
    show_stage('выбор эпохи: модель и данные готовы', started_at)
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_started_at = time.perf_counter()
        fit_one_epoch(model, loader, optimizer, started_at, 'выбор_эпохи: обучение', epoch, MAX_EPOCHS, best_auc)
        probability = positive_probabilities(model, x_outer[inner_idx], started_at, 'выбор_эпохи: валидация', epoch, MAX_EPOCHS, best_auc)
        auc = float(roc_auc_score(y_outer[inner_idx], probability))
        if auc > best_auc:
            best_epoch, best_auc, stale_epochs = epoch, auc, 0
        else:
            stale_epochs += 1
        progress_state['лучший_внутренний_ROC_AUC'] = best_auc
        executed_epochs['выбор_эпохи'] = epoch
        epoch_durations.append(time.perf_counter() - epoch_started_at)
        eta_seconds = float(np.mean(epoch_durations)) * (MAX_EPOCHS - epoch)
        show_progress('выбор эпохи: итог эпохи', epoch, MAX_EPOCHS, started_at, best_auc, estimated_remaining_seconds=eta_seconds)
        if stale_epochs >= PATIENCE:
            show_progress('ранняя остановка', epoch, MAX_EPOCHS, started_at, best_auc, estimated_remaining_seconds=0.0)
            break
    del model, optimizer
    show_progress('выбор эпохи: завершён', executed_epochs['выбор_эпохи'], MAX_EPOCHS, started_at, best_auc, estimated_remaining_seconds=0.0)
    return best_epoch, best_auc


def refit_and_predict(x_train: np.ndarray, y_train: np.ndarray, x_valid: np.ndarray, best_epoch: int, best_auc: float, started_at: float) -> np.ndarray:
    show_progress('дообучение внешнего фолда: подготовка', 0, best_epoch, started_at, best_auc)
    set_fold_seed(FOLD_SEED)
    model = make_model(x_train.shape[1])
    optimizer = make_optimizer(model)
    loader = make_loader(x_train, y_train, FOLD_SEED, shuffle=True)
    show_progress('дообучение внешнего фолда: модель и данные готовы', 0, best_epoch, started_at, best_auc)
    for epoch in range(1, best_epoch + 1):
        fit_one_epoch(model, loader, optimizer, started_at, 'дообучение внешнего фолда', epoch, best_epoch, best_auc)
        executed_epochs['дообучение'] = epoch
        show_progress('дообучение внешнего фолда: итог эпохи', epoch, best_epoch, started_at, best_auc)
    show_progress('прогноз внешней валидации: начало', best_epoch, best_epoch, started_at, best_auc)
    probability = positive_probabilities(model, x_valid, started_at, 'прогноз внешней валидации', best_epoch, best_epoch, best_auc)
    del model, optimizer
    show_progress('прогноз внешней валидации: завершён', best_epoch, best_epoch, started_at, best_auc, estimated_remaining_seconds=0.0)
    return probability


def metrics(y_true: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= 0.5).astype(np.int64)
    auc = float(roc_auc_score(y_true, probability))
    return {
        'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0, 'PR-AUC': float(average_precision_score(y_true, probability)),
        'Precision@0.5': float(precision_score(y_true, predicted, zero_division=0)),
        'Recall@0.5': float(recall_score(y_true, predicted, zero_division=0)),
        'F1@0.5': float(f1_score(y_true, predicted, zero_division=0)),
    }

## Контролируемый диагностический запуск

1. **Что проверяем?** Фактическое время одного полного внешнего фолда и наличие первичных метрик.
2. **Зачем это делаем сейчас?** Это минимально достаточная нагрузка для решения, оправдан ли повторный полный запуск.
3. **Как этот код отвечает на исследовательский вопрос?** Он проверяет SHA-256, воспроизводит рабочую выборку, берёт только фолд №1, обучает TabM, рассчитывает метрики на его внешней валидации и печатает понятный итог.
4. **Что остаётся неизменным?** Закрытая часть разбиения не используется; нет подбора гиперпараметров, обучения GBDT или записи результатов в файловую систему.

In [3]:
started_at = time.perf_counter()
run_status = 'не запущен'
fold_metrics: dict[str, float] | None = None
best_epoch: int | None = None
best_inner_auc: float | None = None
executed_epochs = {'выбор_эпохи': 0, 'дообучение': 0}
progress_state = {'лучший_внутренний_ROC_AUC': None}

try:
    show_stage('проверка SHA-256 данных: начало', started_at)
    if sha256_file(DATASET) != EXPECTED_DATASET_SHA256:
        raise ValueError('SHA-256 набора данных не совпадает с зафиксированным контрактом.')
    show_stage('проверка SHA-256 данных: завершена', started_at)

    show_stage('загрузка данных: начало', started_at)
    data = pd.read_excel(DATASET, engine='pyxlsb')
    show_stage('загрузка данных: завершена', started_at)
    show_stage('проверка схемы и подготовка предикторов: начало', started_at)
    allowed_columns = [column for column in data.columns if column not in [TARGET, IDENTIFIER, *FORBIDDEN_FEATURES]]
    if allowed_columns != FEATURES:
        raise ValueError('Идентичность или порядок разрешённых признаков отличаются от Stage 1.')
    if any(name in FEATURES or name in allowed_columns for name in FORBIDDEN_FEATURES):
        raise ValueError('Запрещённые признаки обнаружены среди предикторов.')

    x_all = data.loc[:, FEATURES].to_numpy(dtype=np.float32)
    y_all = data[TARGET].to_numpy(dtype=np.int64)
    if x_all.shape[1] != 47 or not np.isfinite(x_all).all():
        raise ValueError('Нарушено число признаков либо обнаружены NaN/non-finite значения; imputation не выполняется.')
    show_stage('проверка схемы и подготовка предикторов: завершена', started_at)

    show_stage('воспроизведение рабочего разбиения: начало', started_at)
    all_indices = np.arange(len(data))
    working_indices, holdout_indices = train_test_split(all_indices, test_size=0.20, stratify=y_all, random_state=OUTER_SEED)
    if len(working_indices) != 289614 or len(holdout_indices) != 72404:
        raise ValueError('Размер зафиксированного разбиения не совпадает с контрактом.')
    if hash_indices(working_indices) != EXPECTED_WORKING_INDEX_SHA256:
        raise ValueError('SHA рабочих индексов не совпадает с контрактом Stage 1.')
    x_work, y_work = x_all[working_indices], y_all[working_indices]
    del data, x_all, y_all, all_indices, holdout_indices
    show_stage('воспроизведение рабочего разбиения: завершено', started_at)

    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
    train_idx, valid_idx = next(outer_cv.split(x_work, y_work))
    show_stage('первый внешний фолд: индексы готовы', started_at)
    print({'этап': 'первый внешний фолд: размеры', 'фолд': OUTER_FOLD_NUMBER, 'строк_обучения': len(train_idx), 'строк_валидации': len(valid_idx), 'прошло_минут': round(elapsed_seconds(started_at) / 60.0, 2)}, flush=True)

    best_epoch, best_inner_auc = select_best_epoch(x_work[train_idx], y_work[train_idx], started_at)
    probability = refit_and_predict(x_work[train_idx], y_work[train_idx], x_work[valid_idx], best_epoch, best_inner_auc, started_at)
    show_progress('расчёт метрик первого фолда: начало', best_epoch, best_epoch, started_at, best_inner_auc)
    fold_metrics = metrics(y_work[valid_idx], probability)
    show_progress('расчёт метрик первого фолда: завершён', best_epoch, best_epoch, started_at, best_inner_auc, estimated_remaining_seconds=0.0)
    run_status = 'завершён'

except RuntimeBudgetExceeded:
    run_status = 'превышен двухчасовой лимит'
    show_progress('превышен двухчасовой лимит', None, MAX_EPOCHS, started_at, progress_state['лучший_внутренний_ROC_AUC'], estimated_remaining_seconds=None)

except Exception as error:
    run_status = f'ошибка: {type(error).__name__}'
    print({'этап': 'ошибка', 'фолд': OUTER_FOLD_NUMBER, 'описание': str(error), 'прошло_минут': round(elapsed_seconds(started_at) / 60.0, 2)}, flush=True)

runtime_seconds = elapsed_seconds(started_at)
observed_best_inner_auc = best_inner_auc if best_inner_auc is not None else progress_state['лучший_внутренний_ROC_AUC']
recommend_full_v2 = run_status == 'завершён' and runtime_seconds <= RUNTIME_BUDGET_SECONDS
result_summary = {
    'обучение_завершилось': run_status == 'завершён',
    'статус': run_status,
    'время_минут': round(runtime_seconds / 60.0, 2),
    'эпох_выбора_выполнено': executed_epochs['выбор_эпохи'],
    'эпох_дообучения_выполнено': executed_epochs['дообучение'],
    'всего_эпох_выполнено': sum(executed_epochs.values()),
    'метрики_получены': fold_metrics is not None,
    'метрики_первого_фолда': fold_metrics,
    'лучшая_эпоха': best_epoch,
    'лучший_внутренний_ROC_AUC': observed_best_inner_auc,
    'рекомендация': 'можно переходить к полному эксперименту' if recommend_full_v2 else 'нужно изменить вычислительный план',
}
print('\nИТОГ ПРОВЕРКИ')
print(json.dumps(result_summary, ensure_ascii=False, indent=2))

{'этап': 'проверка SHA-256 данных', 'фолд': 1, 'прошло_сек': 0.0}
{'этап': 'загрузка данных', 'фолд': 1, 'прошло_сек': 0.1}


KeyboardInterrupt: 

## ФАКТЫ

В блоке «ИТОГ ПРОВЕРКИ» выше явно выводятся: завершилось ли обучение, время в минутах, число выполненных эпох выбора и дообучения, их сумма, лучшая эпоха, лучший внутренний ROC-AUC, наличие метрик и рекомендация. До выполнения здесь нет предзаполненного результата.

## ИНТЕРПРЕТАЦИЯ

Если статус «завершён» и метрики получены, облегчённая конфигурация технически выполнима в пределах диагностического лимита. Это основание только для планирования следующего запуска с пересмотром ожидаемого времени; это не доказательство качества TabM относительно базового результата. Если лимит превышен или произошла ошибка, полный запуск в текущем виде не рекомендуется.

## ОГРАНИЧЕНИЯ

Один фолд не оценивает вариативность по фолдам и не даёт статистической значимости. Случайная CV не доказывает временную стабильность. Метрики при пороге 0.5 являются только диагностическими. Проверка не сравнивает TabM с GBDT и не использует закрытую часть данных.

## СЛЕДУЩИЙ ШАГ

При успешном завершении зафиксировать фактическую скорость, оценить масштабирование на три фолда и отдельно утвердить вычислительный бюджет перед полным Этапом 6 V2. При неуспехе сначала уменьшить вычислительную нагрузку или изменить среду выполнения, затем повторить проверку вычислительной реализуемости.